# Validación de la degradación sintética (Fase 1) — Colab

Genera el triple (degradada, GT, máscara) a partir de imágenes limpias y valida:
- la máscara es binaria,
- fuera de la máscara `degradada == GT` (invariante),
- la cobertura de daño es razonable.

El paquete `synthetic_degradation` se importa desde Google Drive
(`MyDrive/TFM/`). Opcionalmente guarda el dataset en Drive con `SAVE_TO_DISK`.

In [ ]:
# Colab: montar Drive y dejar el paquete importable.
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/TFM')   # ajusta si tu carpeta difiere
assert (PROJECT_DIR / 'synthetic_degradation').exists(), \
    f"No encuentro el paquete en {PROJECT_DIR}. Copia synthetic_degradation/ y tests/ a MyDrive/TFM/."
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# opencv y pillow ya vienen en Colab; instala lo que falte sin ruido.
!pip install -q kagglehub
print("Entorno listo.")

In [ ]:
# Re-ejecuta la suite de tests del paquete en el entorno real (Colab).
!cd /content/drive/MyDrive/TFM && python -m pytest tests/ -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from synthetic_degradation import (
    DamageConfig, make_training_sample, mask_coverage, save_sample, generate_dataset,
)

SEED = 1234
cfg = DamageConfig.vintage_base()
print("Config cargada:", cfg)

In [ ]:
# Imágenes limpias = 01_Clean_Candidates_GT del dataset de Kaggle.
# Si la descarga falla, se generan imágenes sintéticas para humo.
def load_clean_images(limit=None) -> list[np.ndarray]:
    clean_dir = None
    try:
        import kagglehub
        root = Path(kagglehub.dataset_download(
            "shrutimandaokar2301/vintage-degraded-image-synthetic-real"))
        hits = list(root.rglob("01_Clean_Candidates_GT"))
        clean_dir = hits[0] if hits else None
    except Exception as e:
        print("kagglehub no disponible o falló:", e)

    paths = []
    if clean_dir and clean_dir.exists():
        for ext in ("*.png", "*.jpg", "*.jpeg"):
            paths.extend(sorted(clean_dir.glob(ext)))
    if not paths:
        print("Sin imágenes limpias; usando sintéticas de prueba.")
        imgs, rng = [], np.random.default_rng(0)
        n = limit if limit is not None else 6
        for _ in range(n):
            base = int(rng.integers(90, 180))
            grad = np.linspace(base - 30, base + 30, 256).astype(np.uint8)
            band = np.repeat(grad[None, :], 256, axis=0)
            imgs.append(np.stack([band, band, band], axis=-1))
        return imgs
    if limit is not None:
        paths = paths[:limit]
    return [np.array(Image.open(p).convert("RGB")) for p in paths]

clean_images = load_clean_images()
print(f"{len(clean_images)} imágenes limpias cargadas.")

In [ ]:
def validate_sample(deg, gt, mask):
    assert set(np.unique(mask)).issubset({0, 255}), "máscara no binaria"
    assert np.array_equal(deg[mask == 0], gt[mask == 0]), "degradada != gt fuera de máscara"
    cov = mask_coverage(mask)
    assert cov < 0.6, f"cobertura sospechosamente alta: {cov:.2%}"
    return cov

rng = np.random.default_rng(SEED)
coverages = []
for i, clean in enumerate(clean_images):
    deg, gt, mask = make_training_sample(clean, rng, cfg)
    cov = validate_sample(deg, gt, mask)
    coverages.append(cov)
print("OK — todas las muestras pasan los asserts.")
print("Cobertura de daño (min/media/max): "
      f"{min(coverages):.2%} / {np.mean(coverages):.2%} / {max(coverages):.2%}")

In [ ]:
n = min(3, len(clean_images))
rng = np.random.default_rng(SEED)  # re-siembra a propósito: muestra las mismas filas 0..n-1 que la validación
fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
if n == 1:
    axes = axes[None, :]
for i in range(n):
    deg, gt, mask = make_training_sample(clean_images[i], rng, cfg)
    overlay = deg.copy()
    overlay[mask > 127] = [255, 0, 0]
    for ax, im, title in zip(
        axes[i],
        [deg, mask, gt, overlay],
        ["Degradada", "Máscara", "GT (limpia, sin daño local)", "Overlay"],
    ):
        ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.show()
print("Verifica: el rojo (máscara) cubre SOLO el daño local, no medias regiones.")

In [ ]:
SAVE_TO_DISK = False                 # ponlo a True para escribir el dataset definitivo
VARIANTS_PER_IMAGE = 5               # nº de degradaciones por imagen limpia
OUT_DIR = PROJECT_DIR / "data" / "lama_synthetic"   # persiste en Google Drive

if SAVE_TO_DISK:
    counts, skipped = generate_dataset(
        clean_images, OUT_DIR, cfg,
        seed=SEED, variants_per_image=VARIANTS_PER_IMAGE)
    total = sum(counts.values())
    print(f"Generadas {total} muestras "
          f"(train={counts['train']}, val={counts['val']}, test={counts['test']}), "
          f"descartadas {skipped}. Salida: {OUT_DIR}")
else:
    print("SAVE_TO_DISK=False — solo validación, no se escribió nada.")